# Methods figures

Regenerates the manuscript's illustrative methods figures using the package's own
functions — the interpolation-limit validation, AMI archetypes, recurrence-plot
readings, and the Procrustes demo. These use synthetic signals so they are fully
self-contained.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

## 1. Interpolation limit: error vs. gap length

A noisy sine (SNR 5:1, 100 Hz, period 1 s) with embedding `m=3, τ=25` gives the
conservative limit `(m−1)τ = 50` samples. We introduce a centred gap of varying
length, linearly interpolate it, and measure the deviation in recurrence rate and
determinism from the gap-free baseline — recovering the manuscript's result that
error accelerates once gaps exceed `(m−1)τ`.

In [ ]:
from pose_dynamics.rqa import RqaParams, run_auto_rqa

fs, T_period = 100.0, 1.0
n = 1500
t = np.arange(n) / fs
clean = np.sin(2 * np.pi * t / T_period)
noise = rng.standard_normal(n)
sig = clean + noise * (clean.std() / 5.0)   # SNR 5:1
m, tau = 3, 25
thresh = (m - 1) * tau                        # = 50 samples

# baseline radius from a target %REC on the gap-free signal, then held fixed
base_p = RqaParams(eDim=m, tLag=tau, radius_mode="fixed_rrec", target_rec=5.0,
                   rescale="mean", norm="zscore", min_line=2)
base = run_auto_rqa(sig, base_p)
fixed_p = RqaParams(eDim=m, tLag=tau, radius_mode="fixed_radius",
                    radius=base.radius_used, rescale="mean", norm="zscore", min_line=2)

gap_mults = np.array([0.5, 1, 2, 3, 4])
rr_err, det_err = [], []
for g in gap_mults * tau:
    g = int(g)
    x = sig.copy()
    c = n // 2
    x[c - g // 2 : c - g // 2 + g] = np.nan
    idx = np.arange(n)
    x = np.interp(idx, idx[np.isfinite(x)], x[np.isfinite(x)])   # linear fill
    r = run_auto_rqa(x, fixed_p)
    rr_err.append(abs(r.metrics["perc_recur"] - base.metrics["perc_recur"]) / base.metrics["perc_recur"])
    det_err.append(abs(r.metrics["perc_determ"] - base.metrics["perc_determ"]) / base.metrics["perc_determ"])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(gap_mults, np.array(rr_err) * 100, "-o", label="Recurrence rate")
ax.plot(gap_mults, np.array(det_err) * 100, "-s", label="Determinism")
ax.axvline((m - 1), color="k", ls="--", label=r"$(m-1)\tau$ threshold")
ax.axhline(5, color="0.6", ls=":", label="5% reference")
ax.set_xlabel(r"gap length (multiples of $\tau$)"); ax.set_ylabel("relative error (%)")
ax.set_title("Interpolation error vs. gap length"); ax.legend();

## 2. AMI archetypes

Average Mutual Information curves for four signal types, computed by the package —
a clear first minimum (rhythmic), a broad plateau (quasi-periodic + noise), an
oscillatory profile (multi-frequency), and a slow decay (oversampled).

In [ ]:
from pose_dynamics.embedding import ami_curve

N = 2000
tt = np.arange(N)
archetypes = {
    "A: clear minimum": np.sin(2 * np.pi * tt / 40) + 0.05 * rng.standard_normal(N),
    "B: broad plateau": np.sin(2 * np.pi * tt / 60) + 0.5 * rng.standard_normal(N),
    "C: oscillatory": (np.sin(2 * np.pi * tt / 40) + np.sin(2 * np.pi * tt / 13)),
    "D: slow decay": np.cumsum(rng.standard_normal(N)) / 30,   # oversampled drift
}
fig, ax = plt.subplots(figsize=(7, 4))
for label, x in archetypes.items():
    c = ami_curve(x, min_lag=1, max_lag=80)
    ax.plot(c.lags, c.ami / c.ami[0], label=label)
ax.set_xlabel(r"delay $\tau$ (frames)"); ax.set_ylabel("AMI (relative to lag 1)")
ax.set_title("AMI archetypes"); ax.legend(fontsize=8);

## 3. Reading recurrence plots

Four canonical dynamics — periodic, a regime shift, laminar (holding) phases, and
chaos — and their recurrence plots (via the package, fixed 3% recurrence rate).

In [ ]:
from pose_dynamics.rqa import RqaParams, run_auto_rqa

Nr = 900
tr = np.linspace(0, 30, Nr)
periodic = np.sin(2 * np.pi * tr / 5)
regime = periodic.copy(); regime[Nr//2:] = 1.5*np.sin(2*np.pi*tr[Nr//2:]/3) + 0.2*rng.standard_normal(Nr-Nr//2)
laminar = periodic.copy()
for s in (150, 350, 600):
    laminar[s:s+80] = laminar[s]
chaotic = np.empty(Nr); chaotic[0] = 0.2
for i in range(Nr-1):
    chaotic[i+1] = 3.9 * chaotic[i] * (1 - chaotic[i])
chaotic = (chaotic - chaotic.mean()) / chaotic.std()

sigs = {"Periodic": periodic, "Regime shift": regime, "Laminar phases": laminar, "Chaotic": chaotic}
p = RqaParams(eDim=3, tLag=4, radius_mode="fixed_rrec", target_rec=3.0, rescale="mean",
              norm="zscore", min_line=2)
fig, axes = plt.subplots(2, 4, figsize=(13, 6.5))
for j, (label, x) in enumerate(sigs.items()):
    res = run_auto_rqa(x, p)
    axes[0, j].plot(x, lw=0.7); axes[0, j].set_title(label, fontsize=10); axes[0, j].set_xticks([])
    res.plot(ax=axes[1, j])
    axes[1, j].set_title(f"%DET={res.metrics['perc_determ']:.0f} LAM={res.metrics['laminarity']:.0f}", fontsize=9)
fig.tight_layout();

## 4. Procrustes alignment demo

A template 2-D skeleton and two participants with different position, rotation, and
scale, aligned to the template by the package's Procrustes.

In [ ]:
from pose_dynamics.features import procrustes_uniform

template = np.array([[0,0],[0,2],[-1,1.5],[1,1.5],[-1,-1],[1,-1],[0,3]], float)  # crude body
xf = lambda th, s, tx, ty: template @ (s * np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])) + np.array([tx, ty])
raw = [xf(0.5, 1.4, 3, 1), xf(-0.4, 0.7, -3, -1)]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for p_ in [template] + raw:
    axes[0].scatter(p_[:,0], p_[:,1])
axes[0].scatter(template[:,0], template[:,1], color="black", s=80, marker="x")
axes[0].set_title("Raw (black x = template)"); axes[0].set_aspect("equal")
axes[1].scatter(template[:,0], template[:,1], color="black", s=80, marker="x", label="template")
for p_ in raw:
    aligned = procrustes_uniform(p_, template, allow_scale=True).apply(p_)
    axes[1].scatter(aligned[:,0], aligned[:,1])
axes[1].set_title("After Procrustes (translation+rotation+scale)"); axes[1].set_aspect("equal"); axes[1].legend()
fig.tight_layout();